# 🚀 LSTM Customers V3 - Training en Kaggle

**Modelo:** Customer Temporal V3 (Forecast 7 días uniforme)

**Configuración:**
- MEDIUM: 120 días → 7 días forecast
- MLflow tracking habilitado
- Bugs corregidos (accuracy, MAE scale)

**GPU:** 2x T4 (Kaggle)

**Tiempo estimado:** 2.5-3 horas

## 📋 INSTRUCCIONES RÁPIDAS

### Antes de ejecutar:

1. **Activar GPU en Kaggle:**
   - Settings (derecha) → Accelerator → **GPU T4 x2**
   - Session timeout → **9 hours**
   - Internet → **On**

2. **Subir archivos al notebook:**
   
   **IMPORTANTE:** Los archivos se suben mediante **"+ Add Data"** (panel derecho)
   
   - Click en **"+ Add Data"**
   - Selecciona pestaña **"Upload"**
   - Arrastra o selecciona estos 3 archivos:
     - `train_all_customers_temporal_3.py` (desde `E:\Codigos\Proyecto Final\src\train\`)
     - `mlflow_tracker.py` (desde `E:\Codigos\Proyecto Final\src\train\`)
     - `online_retail_2.xlsx` (desde `E:\Codigos\Proyecto Final\data\processed\`)
   - Click **"Upload"** y espera a que termine
   
3. **Verificar archivos:**
   - Ejecuta la celda **"2️⃣ Verificar Archivos"**
   - Debe copiar automáticamente los archivos de `/kaggle/input/` a `/kaggle/working/`
   - Si todo está OK, verás: ✅ TODOS LOS ARCHIVOS LISTOS

### Ejecutar:

- **Run All** (o Ctrl+Enter en cada celda)
- Esperar ~2.5-3 horas para el training
- Descargar modelos generados en `/kaggle/working/`

---
## 1️⃣ Setup del Entorno

In [ ]:
%%time
# Verificar GPU
import tensorflow as tf

print("="*70)
print("VERIFICACIÓN DE ENTORNO")
print("="*70)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")
print(f"Número de GPUs: {len(tf.config.list_physical_devices('GPU'))}")

if len(tf.config.list_physical_devices('GPU')) == 0:
    print("\n⚠️ WARNING: No GPU detected!")
    print("   Ve a Settings → Accelerator → GPU T4 x2")
else:
    print("\n✅ GPU configurada correctamente")

In [ ]:
%%time
# Instalar dependencias si es necesario
import sys

print("Instalando MLflow...")
!{sys.executable} -m pip install -q mlflow

print("\n✅ Dependencias instaladas")

---
## 2️⃣ Verificar y Copiar Archivos

**Esta celda:**
1. Busca los archivos subidos en `/kaggle/input/`
2. Los copia automáticamente a `/kaggle/working/`
3. Verifica que todo esté listo

**Re-ejecuta esta celda después de subir archivos.**

In [ ]:
import os
import shutil

print("="*70)
print("VERIFICACIÓN Y COPIA DE ARCHIVOS")
print("="*70)

# Kaggle guarda archivos subidos en /kaggle/input/
# Necesitamos copiarlos a /kaggle/working/

# Buscar archivos en /kaggle/input/
input_dirs = []
if os.path.exists('/kaggle/input'):
    input_dirs = [d for d in os.listdir('/kaggle/input') if os.path.isdir(f'/kaggle/input/{d}')]
    print(f"\n📁 Directorios encontrados en /kaggle/input/: {input_dirs}\n")

required_files = {
    'train_all_customers_temporal_3.py': None,
    'mlflow_tracker.py': None,
    'online_retail_2.xlsx': None
}

# Buscar archivos en todos los directorios de input
for input_dir in input_dirs:
    for root, dirs, files in os.walk(f'/kaggle/input/{input_dir}'):
        for file in files:
            if file in required_files:
                required_files[file] = os.path.join(root, file)

# Verificar y copiar archivos
all_ok = True
for file, source_path in required_files.items():
    if source_path and os.path.exists(source_path):
        dest_path = f'/kaggle/working/{file}'
        shutil.copy2(source_path, dest_path)
        print(f"✅ {file} - copiado a /kaggle/working/")
    else:
        print(f"❌ {file} - NO ENCONTRADO")
        all_ok = False

if not all_ok:
    print("\n" + "="*70)
    print("❌ FALTAN ARCHIVOS!")
    print("="*70)
    print("\n📝 CÓMO SUBIR ARCHIVOS EN KAGGLE:")
    print("\n1. Click en '+ Add Data' (panel derecho)")
    print("2. Selecciona pestaña 'Upload'")
    print("3. Arrastra o selecciona estos archivos:")
    print("   - train_all_customers_temporal_3.py")
    print("   - mlflow_tracker.py")
    print("   - online_retail_2.xlsx")
    print("\n4. Desde estas ubicaciones locales:")
    print("   📂 E:\\Codigos\\Proyecto Final\\src\\train\\")
    print("   📂 E:\\Codigos\\Proyecto Final\\data\\processed\\")
    print("\n5. Click 'Upload' y espera a que termine")
    print("6. Re-ejecuta esta celda")
    print("\n" + "="*70)
else:
    print("\n" + "="*70)
    print("✅ TODOS LOS ARCHIVOS LISTOS EN /kaggle/working/")
    print("="*70)
    print("\n📝 Archivos disponibles:")
    for file in required_files.keys():
        print(f"   ✅ /kaggle/working/{file}")

---
## 3️⃣ Crear Directorios

In [ ]:
%%bash
# Crear estructura de directorios
mkdir -p /kaggle/working/data/processed
mkdir -p /kaggle/working/models/temporal/customer_v3
mkdir -p /kaggle/working/mlruns

echo "✅ Directorios creados:"
ls -la /kaggle/working/

In [ ]:
%%bash
# Mover dataset a la ubicación correcta
if [ -f "/kaggle/working/online_retail_2.xlsx" ]; then
    cp /kaggle/working/online_retail_2.xlsx /kaggle/working/data/processed/
    echo "✅ Dataset copiado a data/processed/"
else
    echo "❌ Dataset no encontrado"
fi

---
## 4️⃣ Modificar Paths en el Script

In [ ]:
# Verificar imports del script
with open('/kaggle/working/train_all_customers_temporal_3.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Verificar que tiene el import de mlflow_tracker
if 'from mlflow_tracker import' in content:
    print("✅ Script tiene imports de MLflow")
else:
    print("⚠️ Script podría no tener MLflow configurado")

# Verificar paths del dataset
if "data/processed/online_retail_2.xlsx" in content:
    print("✅ Path del dataset correcto")
else:
    print("⚠️ Revisar path del dataset")

print("\n📝 Script listo para ejecutar")

---
## 5️⃣ ENTRENAR MODELO - MEDIUM (2.5-3h)

In [ ]:
%%time
# ENTRENAR V3 MEDIUM
import sys
import os

# Cambiar al directorio de trabajo
os.chdir('/kaggle/working')

print("="*70)
print("INICIANDO ENTRENAMIENTO - CUSTOMERS V3 MEDIUM")
print("="*70)
print("Configuración:")
print("  - Horizonte: MEDIUM (120 → 7 días)")
print("  - Platform: kaggle")
print("  - MLflow: Enabled")
print("  - GPU: 2x T4")
print("  - Tiempo estimado: 2.5-3 horas")
print("="*70)
print()

# Ejecutar script con argumentos
!{sys.executable} train_all_customers_temporal_3.py \
    --horizon medium \
    --platform kaggle \
    --script-version v3_kaggle_run1

---
## 6️⃣ Verificar Resultados

In [ ]:
import json
import os

print("="*70)
print("RESULTADOS DEL ENTRENAMIENTO")
print("="*70)

# Buscar archivo de métricas
metrics_path = '/kaggle/working/models/temporal/customer_v3/medium/metrics.json'

if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print("\n✅ MEDIUM - Métricas Finales:")
    print(f"  Accuracy: {metrics['purchase_prob_accuracy']:.2f}%")
    print(f"  AUC: {metrics['purchase_prob_auc']:.4f}")
    print(f"  Days MAE: {metrics['days_mae']:.2f} días")
    print(f"  Value MAE: ${metrics['value_mae']:.2f}")
    print(f"  Total Loss: {metrics['total_loss']:.4f}")
    print(f"  Epochs trained: {metrics['epochs_trained']}")
    print(f"  Train samples: {metrics['train_samples']:,}")
    print(f"  Val samples: {metrics['val_samples']:,}")
else:
    print("\n❌ No se encontraron métricas")
    print(f"   Buscado en: {metrics_path}")

print("\n" + "="*70)

In [ ]:
# Listar archivos generados
print("\n📁 Archivos generados:")
!ls -lh /kaggle/working/models/temporal/customer_v3/medium/

---
## 7️⃣ Visualizar Curvas de Entrenamiento

In [ ]:
from IPython.display import Image, display
import os

# Mostrar gráfica de training history si existe
plot_path = '/kaggle/working/models/temporal/customer_v3/medium/training_history.png'

if os.path.exists(plot_path):
    print("📊 Curvas de Entrenamiento:")
    display(Image(filename=plot_path))
else:
    print("⚠️ Gráfica no encontrada")
    print(f"   Esperado en: {plot_path}")

---
## 8️⃣ Preparar Descarga de Modelos

In [ ]:
%%bash
# Comprimir modelos para descarga
cd /kaggle/working/models/temporal/customer_v3

if [ -d "medium" ]; then
    tar -czf /kaggle/working/customers_v3_medium_kaggle.tar.gz medium/
    echo "✅ Modelos comprimidos en: customers_v3_medium_kaggle.tar.gz"
    ls -lh /kaggle/working/customers_v3_medium_kaggle.tar.gz
else
    echo "❌ Directorio medium/ no encontrado"
fi

---
## 9️⃣ MLflow - Exportar Runs (Opcional)

In [ ]:
import mlflow
import pandas as pd

try:
    # Buscar runs del experimento
    mlflow.set_tracking_uri("/kaggle/working/mlruns")
    experiment = mlflow.get_experiment_by_name("customers_temporal_v3")
    
    if experiment:
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        
        print("\n📊 MLflow Runs encontrados:")
        print(runs[['run_id', 'start_time', 'params.horizon', 'metrics.purchase_prob_accuracy', 'metrics.days_mae']].to_string())
        
        # Exportar a CSV
        runs.to_csv('/kaggle/working/mlflow_runs_v3.csv', index=False)
        print("\n✅ Runs exportados a: mlflow_runs_v3.csv")
    else:
        print("\n⚠️ No se encontró experimento 'customers_temporal_v3'")
        
except Exception as e:
    print(f"\n⚠️ Error accediendo a MLflow: {e}")
    print("   Esto es normal si MLflow no se inicializó correctamente")

---
## 🎯 DESCARGAR ARCHIVOS

### Archivos para descargar:

1. **Modelo comprimido:**
   - `customers_v3_medium_kaggle.tar.gz` (modelo + métricas + historial)

2. **MLflow (opcional):**
   - `mlflow_runs_v3.csv` (tabla de runs)
   - Carpeta `mlruns/` completa (si quieres importar a local)

### Cómo descargar:

1. Click derecho en el archivo en la sidebar izquierda
2. "Download"
3. Guardar en: `E:\Codigos\Proyecto Final\models\temporal\customer_v3\`

---
## 📊 Resumen de Métricas Esperadas

### Customers V3 MEDIUM (120→7d):

| Métrica | Esperado | Notas |
|---------|----------|-------|
| **Accuracy** | 78-90% | Mejor que V2 (forecast más corto) |
| **AUC** | 0.80-0.88 | Buena discriminación |
| **Days MAE** | 10-16 días | Escala real (corregido) |
| **Value MAE** | $35-55 | Escala real (corregido) |
| **Epochs** | ~40-50 | Con early stopping |
| **Samples** | ~1,500 clientes | Depende del dataset |

### Comparación con Products MEDIUM (local):

| Modelo | Forecast | MAE/Accuracy |
|--------|----------|-------------|
| **Products MEDIUM** | 120→7d | MAE: 19.00 unidades |
| **Customers V3 MEDIUM** | 120→7d | Accuracy: ~85% |

**Ambos modelos tienen mismo horizonte → Comparables directamente**

---
## ✅ Checklist Final

Antes de cerrar el notebook:

- [ ] ✅ Entrenamiento completado sin errores
- [ ] ✅ Métricas verificadas (accuracy 78-90%, no >100%)
- [ ] ✅ Days MAE en escala real (10-16 días, NO 0.6)
- [ ] ✅ Modelo comprimido generado
- [ ] ✅ Archivos descargados
- [ ] ✅ MLflow runs exportados (opcional)

### Próximos pasos:

1. **Descargar archivos** de este notebook
2. **Descomprimir** en local: `E:\Codigos\Proyecto Final\models\temporal\customer_v3\`
3. **Iniciar MLflow UI** local: `mlflow ui`
4. **Comparar** Customers V3 vs Products MEDIUM en MLflow
5. **Decidir** configuración final para producción